# HMM for NER — Environment Check
We confirm this notebook is using the correct virtual environment.  
It should point to:  
`C:\Users\murth\Desktop\nlpSession02\codeBase\.venv\Scripts\python.exe`


In [2]:
import sys
import platform

print("Python executable:", sys.executable)
print("Python version   :", platform.python_version())


Python executable: c:\Users\murth\Desktop\nlpSession02\codeBase\.venv\Scripts\python.exe
Python version   : 3.10.11


## Data loading — CoNLL-2003 (BIO tags)

We parse the CoNLL-2003 files from:

`C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003\`

The loader is robust to common filenames:
- `eng.train`, `eng.testa`, `eng.testb` **or**
- `train.txt`, `valid.txt`/`dev.txt`, `test.txt`

Format reminder (space-separated):  
`TOKEN POS CHUNK NER` (e.g., `U.N. NNP I-NP I-ORG`).  
Sentences are separated by blank lines; lines starting with `-DOCSTART-` are ignored.

Outputs:
- `train_sents`, `valid_sents`, `test_sents`: lists of sentences; each sentence is a list of `(token, tag)` pairs.
- Quick stats: #sentences, #tokens, unique tags, and a tiny preview.


In [3]:
from pathlib import Path
from collections import Counter

# --- Paths ---
BASE = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003")

def pick_existing(options):
    """Return the first existing filename from 'options' within BASE."""
    for name in options:
        p = BASE / name
        if p.exists():
            return p
    return None

TRAIN_FILE = pick_existing(["eng.train", "train.txt", "train"])
VALID_FILE = pick_existing(["eng.testa", "valid.txt", "dev.txt", "valid", "dev"])
TEST_FILE  = pick_existing(["eng.testb", "test.txt", "test"])

assert TRAIN_FILE and TRAIN_FILE.exists(), f"Train file not found under {BASE}"
assert VALID_FILE and VALID_FILE.exists(), f"Valid/dev file not found under {BASE}"
assert TEST_FILE  and TEST_FILE.exists(),  f"Test file not found under {BASE}"

def load_conll(path: Path):
    """
    Read a CoNLL-style file. Returns a list of sentences,
    where each sentence is a list of (token, ner_tag).
    """
    sents, sent = [], []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                if sent:
                    sents.append(sent)
                    sent = []
                continue
            if line.startswith("-DOCSTART-"):
                continue
            parts = line.split()
            # Expect at least 2 columns; standard is 4: TOKEN POS CHUNK NER
            token = parts[0]
            tag = parts[-1]  # last column is NER tag
            sent.append((token, tag))
    if sent:
        sents.append(sent)
    return sents

train_sents = load_conll(TRAIN_FILE)
valid_sents = load_conll(VALID_FILE)
test_sents  = load_conll(TEST_FILE)

# --- Quick stats ---
def stats(name, sents):
    n_sents = len(sents)
    n_toks  = sum(len(s) for s in sents)
    tags = Counter(tag for s in sents for _, tag in s)
    return name, n_sents, n_toks, len(tags), tags.most_common(5)

info = [
    stats("train", train_sents),
    stats("valid", valid_sents),
    stats("test",  test_sents),
]

print("Files:")
print("  Train:", TRAIN_FILE)
print("  Valid:", VALID_FILE)
print("  Test :", TEST_FILE)
print("\nDataset stats (name, #sentences, #tokens, #unique_tags, top5_tags):")
for row in info:
    print(" ", row)

# Tiny preview
for name, sents in [("train", train_sents), ("valid", valid_sents), ("test", test_sents)]:
    print(f"\n{name} preview (first sentence):")
    for tok, tag in (sents[0][:12] if sents else []):
        print(f"{tok:15s} {tag}")
    if sents and len(sents[0]) > 12:
        print("... (truncated)")


Files:
  Train: C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003\train.txt
  Valid: C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003\valid.txt
  Test : C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003\test.txt

Dataset stats (name, #sentences, #tokens, #unique_tags, top5_tags):
  ('train', 14041, 203621, 9, [('O', 169578), ('B-LOC', 7140), ('B-PER', 6600), ('B-ORG', 6321), ('I-PER', 4528)])
  ('valid', 3250, 51362, 9, [('O', 42759), ('B-PER', 1842), ('B-LOC', 1837), ('B-ORG', 1341), ('I-PER', 1307)])
  ('test', 3453, 46435, 9, [('O', 38323), ('B-LOC', 1668), ('B-ORG', 1661), ('B-PER', 1617), ('I-PER', 1156)])

train preview (first sentence):
EU              B-ORG
rejects         O
German          B-MISC
call            O
to              O
boycott         O
British         B-MISC
lamb            O
.               O

valid preview (first sentence):
CRICKET         O
-               O
LEICESTERSHIRE  B-ORG
TAKE            O
OVER            O
AT            

## Vocab & Tagset (with `<UNK>` policy)

We build:
- **Tagset** from training BIO tags.
- **Word vocab** from training tokens only.
- Reserve IDs: `<PAD>=0`, `<UNK>=1`. Rare/unknown words → `<UNK>`.

**UNK policy:** any token with train frequency ≤ 1 is folded into `<UNK>`.  
We then compute **OOV rates** on valid/test to sanity-check coverage.


In [4]:
from collections import Counter, defaultdict

PAD, UNK = "<PAD>", "<UNK>"

# --- Tagset from train ---
train_tags = [tag for sent in train_sents for _, tag in sent]
tag_list = sorted(set(train_tags))  # BIO tags seen in train
tag2id = {t:i for i,t in enumerate(tag_list)}
id2tag = {i:t for t,i in tag2id.items()}

print(f"#Tags: {len(tag_list)} -> {tag_list}")

# --- Word vocab from train with UNK threshold ---
train_tokens = [tok for sent in train_sents for tok, _ in sent]
freq = Counter(train_tokens)

# Threshold ≤1 -> UNK
def build_word_vocab(counter: Counter, unk_thresh: int = 1):
    vocab = [PAD, UNK]
    for w,c in counter.items():
        if c > unk_thresh:
            vocab.append(w)
    word2id = {w:i for i,w in enumerate(vocab)}
    id2word = {i:w for w,i in word2id.items()}
    return word2id, id2word

word2id, id2word = build_word_vocab(freq, unk_thresh=1)
pad_id, unk_id = word2id[PAD], word2id[UNK]

print(f"#Vocab (incl PAD/UNK): {len(word2id)}")
print(f"PAD id = {pad_id}, UNK id = {unk_id}")

# --- Helpers: maps & OOV rate ---
def tok2id(tok: str) -> int:
    return word2id.get(tok, unk_id)

def sent2ids(sent):
    xs = [tok2id(tok) for tok, _ in sent]
    ys = [tag2id[tag] for _, tag in sent]
    return xs, ys

def corpus_oov_rate(sents):
    total = 0
    oov = 0
    for sent in sents:
        for tok, _ in sent:
            total += 1
            if tok not in word2id:
                oov += 1
    return (oov / max(total,1)) * 100.0, oov, total

valid_oov_pct, v_oov, v_tot = corpus_oov_rate(valid_sents)
test_oov_pct,  t_oov, t_tot = corpus_oov_rate(test_sents)

print(f"OOV (valid): {valid_oov_pct:.2f}%  ({v_oov}/{v_tot})")
print(f"OOV (test) : {test_oov_pct:.2f}%  ({t_oov}/{t_tot})")

# Quick preview: map first valid sentence to ids
x_ids, y_ids = sent2ids(valid_sents[0])
print("\nPreview (valid[0]) token→id (first 12):")
for (tok,_), xi in list(zip(valid_sents[0], x_ids))[:12]:
    print(f"{tok:15s} -> {xi}")


#Tags: 9 -> ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
#Vocab (incl PAD/UNK): 11984
PAD id = 0, UNK id = 1
OOV (valid): 11.63%  (5975/51362)
OOV (test) : 15.57%  (7231/46435)

Preview (valid[0]) token→id (first 12):
CRICKET         -> 1846
-               -> 593
LEICESTERSHIRE  -> 1
TAKE            -> 5456
OVER            -> 1809
AT              -> 1489
TOP             -> 9371
AFTER           -> 2002
INNINGS         -> 1
VICTORY         -> 9093
.               -> 9


## Supervised HMM parameters from CoNLL-2003 (smoothed MLE)

We estimate HMM params from **labeled** train data:

- **Initial** `π[i]`: P(tag\_0 = i) from first tag of each train sentence.
- **Transitions** `A[i,j]`: P(tag\_t = j | tag\_{t-1} = i) from tag bigrams inside sentences.
- **Emissions** `B[i,w]`: P(word\_t = w | tag\_t = i) from tag–word pairs; rare/unknown words map to `<UNK>`.

All with **add-α smoothing** to avoid zeros:
- π: απ = 0.5  
- A: αA = 0.5  
- B: αB = 0.5  

We keep a **log-space** copy for numerically-stable decoding later. Sanity checks confirm each row sums ≈ 1 in probability space.


In [5]:
import numpy as np
from collections import defaultdict

T = len(tag_list)
V = len(word2id)

alpha_pi = 0.5
alpha_A  = 0.5
alpha_B  = 0.5

# --- Count tables ---
pi_counts   = np.zeros(T, dtype=np.float64)
A_counts    = np.zeros((T, T), dtype=np.float64)
B_counts    = np.zeros((T, V), dtype=np.float64)

# Helper: get ids (uses earlier word/tag maps)
def tags_of_sentence(sent):
    return [tag2id[tag] for _, tag in sent]

def words_of_sentence(sent):
    return [tok2id(tok) for tok, _ in sent]

# Populate counts from train_sents
for sent in train_sents:
    tag_ids = tags_of_sentence(sent)
    word_ids = words_of_sentence(sent)
    if not tag_ids:
        continue

    # initial tag
    pi_counts[tag_ids[0]] += 1.0

    # emissions
    for t, w in zip(tag_ids, word_ids):
        B_counts[t, w] += 1.0

    # transitions (within sentence)
    for i, j in zip(tag_ids[:-1], tag_ids[1:]):
        A_counts[i, j] += 1.0

# --- Add-alpha smoothing & normalize ---
def normalize_with_add_alpha(counts, alpha, axis):
    counts = counts + alpha
    denom = counts.sum(axis=axis, keepdims=True)
    probs = counts / np.clip(denom, 1e-12, None)
    return probs

# π (shape: [T])
pi_probs = (pi_counts + alpha_pi) / (pi_counts.sum() + alpha_pi * T)

# A (shape: [T, T]) row-stochastic
A_probs = normalize_with_add_alpha(A_counts, alpha_A, axis=1)

# B (shape: [T, V]) row-stochastic
B_probs = normalize_with_add_alpha(B_counts, alpha_B, axis=1)

# --- Log-space copies (avoid log(0)) ---
def safe_log(x):
    return np.log(np.clip(x, 1e-300, None))

log_pi = safe_log(pi_probs)
log_A  = safe_log(A_probs)
log_B  = safe_log(B_probs)

# --- Sanity checks ---
def row_sums(mat):
    return mat.sum(axis=1)

print("π sum:", float(pi_probs.sum()))
print("A row sums: min/max =", row_sums(A_probs).min(), row_sums(A_probs).max())
print("B row sums: min/max =", row_sums(B_probs).min(), row_sums(B_probs).max())

# Tiny peek at a couple of tags
sample_tags = tag_list[:5]
print("\nSample tags:", sample_tags)
for tname in sample_tags:
    tid = tag2id[tname]
    # show top-5 emission words by probability (skip PAD)
    top5 = np.argsort(-B_probs[tid])[:6]  # larger slice then filter PAD if present
    top5 = [w for w in top5 if w != word2id[PAD]][:5]
    pairs = [(id2word[w], float(B_probs[tid, w])) for w in top5]
    print(f"Top emissions for tag {tname}: {pairs[:5]}")


π sum: 1.0
A row sums: min/max = 1.0 1.0
B row sums: min/max = 0.9999999999999998 1.0000000000000002

Sample tags: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC']
Top emissions for tag B-LOC: [('<UNK>', 0.03727535790435577), ('U.S.', 0.02356838257691136), ('Germany', 0.010775205604629912), ('Britain', 0.010166006701187938), ('Australia', 0.009937557112397197)]
Top emissions for tag B-MISC: [('<UNK>', 0.027836691410392366), ('Russian', 0.010127253446447508), ('German', 0.008960763520678684), ('British', 0.008324496288441145), ('French', 0.0071580063626723225)]
Top emissions for tag B-ORG: [('<UNK>', 0.04202875010151872), ('Reuters', 0.006456590595305774), ('St', 0.00353285145780882), ('U.N.', 0.0032892065296840738), ('New', 0.0032079915536424915)]
Top emissions for tag B-PER: [('<UNK>', 0.06579574332909784), ('Clinton', 0.0055987928843710295), ('Mark', 0.004645806861499365), ('Michael', 0.004407560355781448), ('David', 0.004010482846251588)]
Top emissions for tag I-LOC: [('<UNK>', 0.0129

## Viterbi decoding — best tag path per sentence (log-space)

We implement Viterbi using our `log_pi`, `log_A`, `log_B`:

For a sentence \(x_{1..T}\) and tags \(y_{1..T}\),
\[
\arg\max_{y_{1..T}} \big( \log \pi_{y_1} + \log B_{y_1, x_1} + \sum_{t=2}^{T} (\log A_{y_{t-1}, y_t} + \log B_{y_t, x_t}) \big)
\]

We keep:
- `delta[t, j]`: best log-score for time `t` ending in tag `j`
- `psi[t, j]`: backpointer to the best previous tag index

Returns the best tag id path and its log-score. A corpus-level helper decodes all sentences.


In [6]:
import numpy as np

def viterbi_decode(word_ids, log_pi, log_A, log_B):
    """
    word_ids: List[int] for one sentence
    log_pi:   (T,)
    log_A:    (T, T)
    log_B:    (T, V)
    Returns: (best_path_ids: List[int], best_logscore: float)
    """
    T_tags = log_A.shape[0]
    L = len(word_ids)
    if L == 0:
        return [], 0.0

    # DP tables
    delta = np.full((L, T_tags), -np.inf, dtype=np.float64)
    psi   = np.full((L, T_tags), -1, dtype=np.int32)

    # t = 0 (init)
    w0 = word_ids[0]
    delta[0] = log_pi + log_B[:, w0]

    # t = 1..L-1
    for t in range(1, L):
        wt = word_ids[t]
        # For each current tag j, we consider all prev tags i:
        # score(i->j) = delta[t-1, i] + log_A[i, j] + log_B[j, wt]
        # We can vectorize over i for each j
        prev = delta[t-1][:, None] + log_A  # shape (T_tags, T_tags)
        psi[t]   = np.argmax(prev, axis=0)
        delta[t] = prev[psi[t], np.arange(T_tags)] + log_B[:, wt]

    # Termination
    last_tag = int(np.argmax(delta[L-1]))
    best_logscore = float(delta[L-1, last_tag])

    # Backtrack
    best_path = [last_tag]
    for t in range(L-1, 0, -1):
        last_tag = int(psi[t, last_tag])
        best_path.append(last_tag)
    best_path.reverse()
    return best_path, best_logscore

def decode_sentence(sent):
    xs = [tok2id(tok) for tok, _ in sent]
    path, score = viterbi_decode(xs, log_pi, log_A, log_B)
    tags = [id2tag[i] for i in path]
    return tags, score

def decode_corpus(sents, max_items=None):
    preds = []
    scores = []
    for i, sent in enumerate(sents):
        if max_items is not None and i >= max_items:
            break
        tags, score = decode_sentence(sent)
        preds.append(tags)
        scores.append(score)
    return preds, scores

# Quick smoke test on 3 sentences from valid set
valid_preds, valid_scores = decode_corpus(valid_sents, max_items=3)
for k in range(len(valid_preds)):
    print(f"\nSentence {k} (len={len(valid_sents[k])}) best log-score: {valid_scores[k]:.2f}")
    print("TOK".ljust(18), "PRED".ljust(10), "GOLD")
    for (tok, gold), pred in zip(valid_sents[k][:20], valid_preds[k][:20]):
        print(f"{tok[:16]:18s} {pred:10s} {gold}")
    if len(valid_sents[k]) > 20:
        print("... (truncated)")



Sentence 0 (len=11) best log-score: -85.16
TOK                PRED       GOLD
CRICKET            O          O
-                  O          O
LEICESTERSHIRE     O          B-ORG
TAKE               O          O
OVER               O          O
AT                 O          O
TOP                O          O
AFTER              O          O
INNINGS            O          O
VICTORY            O          O
.                  O          O

Sentence 1 (len=2) best log-score: -17.12
TOK                PRED       GOLD
LONDON             B-LOC      B-LOC
1996-08-30         O          O

Sentence 2 (len=35) best log-score: -249.59
TOK                PRED       GOLD
West               B-LOC      B-MISC
Indian             I-LOC      I-MISC
all-rounder        O          O
Phil               B-PER      B-PER
Simmons            I-PER      I-PER
took               O          O
four               O          O
for                O          O
38                 O          O
on                 O          O
F

## Quick metric — token-level accuracy on the validation set

Before full CoNLL span-level F1, do a fast sanity check:
- Run Viterbi on **all** validation sentences.
- Compare predicted tag **per token** with the gold tag.
- Report overall **token accuracy** and a small confusion summary (top mismatches).

*Requires:* previous blocks executed (`valid_sents`, `tag2id/id2tag`, `tok2id`, `log_pi`, `log_A`, `log_B`, `decode_sentence`).


In [7]:
from collections import Counter

def token_accuracy(sents):
    """Return accuracy, total tokens, and a Counter of (gold,pred) confusions."""
    correct = 0
    total = 0
    conf = Counter()
    for sent in sents:
        pred_tags, _ = decode_sentence(sent)
        gold_tags = [tag for _, tag in sent]
        # align (should match lengths; if not, clip safely)
        L = min(len(pred_tags), len(gold_tags))
        for g, p in zip(gold_tags[:L], pred_tags[:L]):
            total += 1
            correct += int(g == p)
            if g != p:
                conf[(g, p)] += 1
    acc = correct / max(total, 1)
    return acc, total, conf

valid_acc, valid_total, valid_conf = token_accuracy(valid_sents)
print(f"Valid token accuracy: {valid_acc*100:.2f}%  ({valid_total} tokens)")

# Show top-10 confusions (gold -> pred)
print("\nTop 10 tag confusions (gold -> pred, count):")
for (g, p), c in valid_conf.most_common(10):
    print(f"{g:8s} -> {p:8s} : {c}")


Valid token accuracy: 94.23%  (51362 tokens)

Top 10 tag confusions (gold -> pred, count):
B-PER    -> O        : 649
I-PER    -> O        : 389
B-ORG    -> O        : 316
B-LOC    -> O        : 301
I-ORG    -> O        : 193
B-MISC   -> O        : 166
I-MISC   -> O        : 105
B-LOC    -> B-ORG    : 87
B-ORG    -> B-LOC    : 63
O        -> I-MISC   : 62


## Span-level NER metrics — CoNLL-style Precision / Recall / F1

We evaluate **entities** (exact span + type match) from BIO tags.

Rules:
- Begin a span on `B-X`; continue with `I-X`; anything else ends current span.
- A predicted span counts **correct** only if **boundaries and type** match the gold.

Outputs:
- **Micro** P/R/F1 over all entity types.
- Per-type P/R/F1 (PER, ORG, LOC, MISC).


In [8]:
from collections import Counter, defaultdict
from typing import List, Tuple

def bio_spans(tags: List[str]) -> List[Tuple[str,int,int]]:
    """
    Convert BIO tag sequence to spans.
    Returns list of (TYPE, start, end) with end-exclusive.
    """
    spans = []
    cur_type, start = None, None
    for i, t in enumerate(tags + ["O"]):  # sentinel O to flush
        if t == "O" or t.startswith("B-"):
            # close previous
            if cur_type is not None:
                spans.append((cur_type, start, i))
                cur_type, start = None, None
            # start new?
            if t.startswith("B-"):
                cur_type = t[2:]
                start = i
        elif t.startswith("I-"):
            ttype = t[2:]
            if cur_type is None or ttype != cur_type:
                # Invalid I- continuation: treat as B-
                if cur_type is not None:
                    spans.append((cur_type, start, i))
                cur_type, start = ttype, i
        else:
            # Unknown label, treat as O
            if cur_type is not None:
                spans.append((cur_type, start, i))
                cur_type, start = None, None
    return spans

def eval_spans(gold_sents, decode_fn):
    gold_total = 0
    pred_total = 0
    correct    = 0

    per_type = defaultdict(lambda: {"gold":0, "pred":0, "correct":0})

    for sent in gold_sents:
        gold_tags = [tag for _, tag in sent]
        pred_tags, _ = decode_fn(sent)

        gold_sp = bio_spans(gold_tags)
        pred_sp = bio_spans(pred_tags)

        gold_set = set(gold_sp)
        pred_set = set(pred_sp)

        gold_total += len(gold_sp)
        pred_total += len(pred_sp)
        correct_now = len(gold_set & pred_set)
        correct    += correct_now

        # per-type accounting
        for t,_,_ in gold_sp:
            per_type[t]["gold"] += 1
        for t,_,_ in pred_sp:
            per_type[t]["pred"] += 1
        for t,_,_ in (gold_set & pred_set):
            per_type[t]["correct"] += 1

    def prf(c,p,g):
        P = c / p if p else 0.0
        R = c / g if g else 0.0
        F = 2*P*R/(P+R) if (P+R) else 0.0
        return P,R,F

    P,R,F = prf(correct, pred_total, gold_total)
    print(f"[MICRO] P={P*100:.2f}  R={R*100:.2f}  F1={F*100:.2f}  "
          f"(gold={gold_total}, pred={pred_total}, correct={correct})")

    # Per-type table (sorted by gold count desc)
    rows = []
    for t, d in per_type.items():
        p,r,f = prf(d["correct"], d["pred"], d["gold"])
        rows.append((t, d["gold"], d["pred"], d["correct"], p, r, f))
    rows.sort(key=lambda x: (-x[1], x[0]))
    print("\nPer-type metrics:")
    print(f"{'TYPE':6s} {'GOLD':>6s} {'PRED':>6s} {'CORR':>6s} {'P%':>7s} {'R%':>7s} {'F1%':>7s}")
    for t,g,p,c,pp,rr,ff in rows:
        print(f"{t:6s} {g:6d} {p:6d} {c:6d} {pp*100:7.2f} {rr*100:7.2f} {ff*100:7.2f}")

# Run on validation set
print("Validation (span-level, strict match):")
eval_spans(valid_sents, decode_sentence)


Validation (span-level, strict match):
[MICRO] P=85.25  R=66.54  F1=74.74  (gold=5942, pred=4638, correct=3954)

Per-type metrics:
TYPE     GOLD   PRED   CORR      P%      R%     F1%
PER      1842   1180   1047   88.73   56.84   69.29
LOC      1837   1570   1409   89.75   76.70   82.71
ORG      1341   1127    841   74.62   62.71   68.15
MISC      922    761    657   86.33   71.26   78.07
